# Question 1

In [0]:
%sql
SELECT
    'US population 2013-2018' AS label,
    AVG(population) AS average_pop,
    STDDEV_POP(population) AS stddev_pop
FROM labor_stats.silver.population
WHERE year BETWEEN 2013 AND 2018

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
res = (
    spark.table("labor_stats.silver.population")
    .filter(F.col("year").between(2013, 2018))
    .agg(
        F.avg("population").alias("average_pop"),
        F.stddev_pop("population").alias("stddev_pop"),
    )
    .withColumn("label", F.lit("US population 2013-2018"))
    .select("label", "average_pop", "stddev_pop")
)

display(res)

# Question 2 

In [0]:
%sql

WITH quarterly_series AS (
    SELECT DISTINCT series_id
    FROM labor_stats.silver.pr_data_1_alldata
    WHERE period IN ('Q01', 'Q02', 'Q03', 'Q04')
),
quarterly_years AS (
    SELECT
        series_id,
        year,
        'quarter_value_sum' AS value_type,
        SUM(value) AS value
    FROM labor_stats.silver.pr_data_1_alldata
    WHERE period IN ('Q01', 'Q02', 'Q03', 'Q04')
      AND value IS NOT NULL
    GROUP BY series_id, year
),
annual_only_years AS (
    SELECT
        observations.series_id,
        observations.year,
        'annual_average' AS value_type,
        observations.value
    FROM labor_stats.silver.pr_data_1_alldata AS observations
    LEFT ANTI JOIN quarterly_series
      ON observations.series_id = quarterly_series.series_id
    WHERE observations.period = 'Q05'
      AND observations.value IS NOT NULL
),
candidate_years AS (
    SELECT * FROM quarterly_years
    UNION ALL
    SELECT * FROM annual_only_years
),
best_year AS (
    SELECT
        series_id,
        year,
        value_type,
        value,
        ROW_NUMBER() OVER (
            PARTITION BY series_id
            ORDER BY value DESC, year DESC
        ) AS rnk
    FROM candidate_years
),

labels AS (
    SELECT
        series.series_id,
        CONCAT_WS(
            ' | ',
            sector.sector_name,
            worker_class.class_text,
            measure.measure_text,
            duration.duration_text
        ) AS series_label
    FROM labor_stats.silver.pr_series AS series
    LEFT JOIN labor_stats.silver.pr_sector AS sector
      ON series.sector_code = sector.sector_code
    LEFT JOIN labor_stats.silver.pr_class AS worker_class
      ON series.class_code = worker_class.class_code
    LEFT JOIN labor_stats.silver.pr_measure AS measure
      ON series.measure_code = measure.measure_code
    LEFT JOIN labor_stats.silver.pr_duration AS duration
      ON series.duration_code = duration.duration_code
)

SELECT
    labels.series_id,
    labels.series_label,
    best_year.year AS best_year,
    best_year.value_type,
    best_year.value
FROM labels
INNER JOIN best_year
  ON labels.series_id = best_year.series_id
WHERE best_year.rnk = 1

In [0]:
QUARTERS = ("Q01", "Q02", "Q03", "Q04")

def silver(name):
    return spark.table(f"labor_stats.silver.{name}")

rank = Window.partitionBy("series_id").orderBy(
    F.desc("value"),
    F.desc("year"),
)

observations = silver("pr_data_1_alldata")
quarterly_series = (
    observations.where(F.col("period").isin(*QUARTERS))
    .select("series_id")
    .distinct()
)

quarterly_years = (
    observations
    .where(
        F.col("period").isin(*QUARTERS)
        & F.col("value").isNotNull()
    )
    .groupBy("series_id", "year")
    .agg(F.sum("value").alias("quarter_value_sum"))
    .select(
        "series_id",
        "year",
        F.lit("quarter_value_sum").alias("value_type"),
        F.col("quarter_value_sum").alias("value"),
    )
)

annual_only_years = (
    observations.where(
        (F.col("period") == "Q05")
        & F.col("value").isNotNull()
    )
    .join(quarterly_series, "series_id", "left_anti")
    .select(
        "series_id",
        "year",
        F.lit("annual_average").alias("value_type"),
        "value",
    )
)

best_year = (
    quarterly_years.unionByName(annual_only_years)
    .withColumn("rnk", F.row_number().over(rank))
    .where("rnk = 1")
)

labels = (
    silver("pr_series")
    .join(silver("pr_sector"), "sector_code", "left")
    .join(silver("pr_class"), "class_code", "left")
    .join(silver("pr_measure"), "measure_code", "left")
    .join(silver("pr_duration"), "duration_code", "left")
    .select(
        "series_id",
        F.concat_ws(
            " | ",
            "sector_name",
            "class_text",
            "measure_text",
            "duration_text",
        ).alias("series_label"),
    )
)

result = (
    labels.join(best_year, "series_id", "inner")
    .select(
        "series_id",
        "series_label",
        F.col("year").alias("best_year"),
        "value_type",
        "value",
    )
)

display(result)

In [0]:
%sql
SELECT *
FROM labor_stats.silver.pr_period


# Question 3

In [0]:
%sql
SELECT
    a.value,
    a.series_id,
    a.period,
    a.year,
    b.population
FROM labor_stats.silver.pr_data_1_alldata as a
LEFT JOIN labor_stats.silver.population as b
  ON a.year = b.year
WHERE a.period = 'Q01'
  AND a.series_id = 'PRS30006032'

In [0]:
observations = spark.table("labor_stats.silver.pr_data_1_alldata").alias("observations")
population = spark.table("labor_stats.silver.population").alias("population")

res = (
    observations.join(
        population,
        F.col("observations.year") == F.col("population.year"),
        "left",
    )
    .filter(
        (F.col("observations.period") == "Q01")
        & (F.col("observations.series_id") == "PRS30006032")
    )
    .select(
        F.col("observations.value").alias("value"),
        F.col("observations.series_id").alias("series_id"),
        F.col("observations.period").alias("period"),
        F.col("observations.year").alias("year"),
        F.col("population.population").alias("population"),
    )
)

display(res)